# OpenOppsDB hiring market map

This topical notebook maps current open roles by company, provider, location, and remote/workplace signal.


In [ ]:
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd

db_candidates = sorted(Path("/kaggle/input").glob("**/openoppsdb.sqlite"))
if not db_candidates:
    raise FileNotFoundError("No openoppsdb.sqlite input found under /kaggle/input")
DB_PATH = db_candidates[0]
DATASET_DIR = DB_PATH.parent
DB_URI = f"file:{DB_PATH}?mode=ro&immutable=1"
print(f"Reading OpenOppsDB snapshot from {DB_PATH}")


In [ ]:
with sqlite3.connect(DB_URI, uri=True) as conn:
    top_companies = pd.read_sql_query(
        """
        select
            coalesce(v.company, b.name, 'Unknown') as company,
            count(*) as open_roles,
            count(distinct j.board_key) as boards
        from jobs j
        join job_versions v on v.id = j.current_version_id
        left join boards b on b.key = j.board_key
        where j.status = 'open'
        group by company
        order by open_roles desc, company
        limit 20
        """,
        conn,
    )
    provider_mix = pd.read_sql_query(
        """
        select provider_id, count(*) as open_roles
        from jobs
        where status = 'open'
        group by provider_id
        order by open_roles desc, provider_id
        limit 15
        """,
        conn,
    )
    location_mix = pd.read_sql_query(
        """
        select l.label as location, count(distinct j.id) as open_roles
        from jobs j
        join job_versions v on v.id = j.current_version_id
        join job_version_locations l on l.job_version_id = v.id
        where j.status = 'open' and l.label is not null and l.label <> ''
        group by l.label
        order by open_roles desc, location
        limit 20
        """,
        conn,
    )
    remote_mix = pd.read_sql_query(
        """
        select coalesce(v.remote, 'Unknown') as remote, count(*) as open_roles
        from jobs j
        join job_versions v on v.id = j.current_version_id
        where j.status = 'open'
        group by remote
        order by open_roles desc
        """,
        conn,
    )

display(top_companies)
display(provider_mix)
display(location_mix)
remote_mix


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if not top_companies.empty:
    top_companies.head(10).sort_values("open_roles").plot.barh(
        x="company", y="open_roles", ax=axes[0], legend=False, title="Top companies"
    )
else:
    axes[0].set_title("Top companies")
if not location_mix.empty:
    location_mix.head(10).sort_values("open_roles").plot.barh(
        x="location", y="open_roles", ax=axes[1], legend=False, title="Top locations"
    )
else:
    axes[1].set_title("Top locations")
plt.tight_layout()
plt.show()
